Bronze table = raw ingested data
Silver table = cleaned, named, typed, class labels converted, duplicate flag added, missing values checked

In [0]:
# ============================================================
# MAGIC GAMMA TELESCOPE PROJECT
# BRONZE TO SILVER - DATABRICKS / PYSPARK
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

In [0]:
display(spark.table("workspace.medallion_data.bronze_telescope"))

In [0]:
# ============================================================
# 0. LOAD BRONZE TABLE
# GAMMA TELESCOPE
# Source: Raw → Bronze notebook created by teammate
# ============================================================

bronze_table_name = "workspace.medallion_data.bronze_telescope"

bronze_df = spark.table(bronze_table_name)

display(bronze_df.limit(10))
bronze_df.printSchema()

print("Bronze row count:", bronze_df.count())
print("Bronze column count:", len(bronze_df.columns))

In [0]:
# ============================================================
# 1. Load Bronze table
# ============================================================

bronze_df = spark.table("workspace.medallion_data.bronze_telescope")

display(bronze_df.limit(10))
bronze_df.printSchema()

print("Bronze row count:", bronze_df.count())
print("Bronze column count:", len(bronze_df.columns))

In [0]:
# ============================================================
# 2. Rename columns
# ============================================================

silver_columns = [
    "f_length",
    "f_width",
    "f_size",
    "f_conc",
    "f_conc1",
    "f_asym",
    "f_m3long",
    "f_m3trans",
    "f_alpha",
    "f_dist",
    "class"
]

silver_df = bronze_df.toDF(*silver_columns)

display(silver_df)
silver_df.printSchema()

In [0]:
# ============================================================
# 3. Cast feature columns to numeric
# ============================================================

feature_columns = [
    "f_length",
    "f_width",
    "f_size",
    "f_conc",
    "f_conc1",
    "f_asym",
    "f_m3long",
    "f_m3trans",
    "f_alpha",
    "f_dist"
]

for col_name in feature_columns:
    silver_df = silver_df.withColumn(
        col_name,
        F.col(col_name).cast(DoubleType())
    )

silver_df.printSchema()
display(silver_df)

In [0]:
# ============================================================
# 4. Convert class labels from g/h to Gamma/Hadron
# ============================================================

silver_df = silver_df.withColumn(
    "class",
    F.when(F.col("class") == "g", "Gamma")
     .when(F.col("class") == "h", "Hadron")
     .otherwise(F.col("class"))
)

display(silver_df.groupBy("class").count())

In [0]:
# ============================================================
# 5. Add row_id to track original observations
# ============================================================

window_spec = Window.orderBy(F.monotonically_increasing_id())

silver_df = silver_df.withColumn(
    "row_id",
    F.row_number().over(window_spec)
)

display(silver_df)

In [0]:
# ============================================================
# 6. Check dimensions
# ============================================================

row_count = silver_df.count()
column_count = len(silver_df.columns)

print("Silver row count:", row_count)
print("Silver column count:", column_count)

In [0]:
# ============================================================
# 7. Missing value summary
# ============================================================

missing_summary = silver_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silver_df.columns
])

display(missing_summary)

# ============================================================
# 7B. Missing value summary in long format
# ============================================================

missing_counts = []

for c in silver_df.columns:
    missing_count = silver_df.filter(F.col(c).isNull()).count()
    missing_counts.append((c, missing_count))

missing_df = spark.createDataFrame(
    missing_counts,
    ["column_name", "missing_count"]
)

display(missing_df)

In [0]:
# ============================================================
# 8. Check exact duplicates
# ============================================================

duplicate_check_columns = [
    c for c in silver_df.columns if c != "row_id"
]

total_rows = silver_df.count()
distinct_rows = silver_df.select(duplicate_check_columns).distinct().count()
duplicate_rows_count = total_rows - distinct_rows

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicate rows:", duplicate_rows_count)

In [0]:
# ============================================================
# 9. Identify duplicate groups
# ============================================================

duplicate_report = (
    silver_df
    .groupBy(duplicate_check_columns)
    .agg(F.count("*").alias("duplicate_count"))
    .filter(F.col("duplicate_count") > 1)
    .orderBy(F.desc("duplicate_count"))
)

display(duplicate_report)

print("Number of duplicate groups:", duplicate_report.count())

In [0]:
# ============================================================
# 10. Add duplicate_count and duplicate flag to Silver table
# ============================================================

duplicate_counts = (
    silver_df
    .groupBy(duplicate_check_columns)
    .agg(F.count("*").alias("duplicate_count"))
)

silver_df = (
    silver_df
    .join(
        duplicate_counts,
        on=duplicate_check_columns,
        how="left"
    )
    .withColumn(
        "is_duplicate_group",
        F.when(F.col("duplicate_count") > 1, True).otherwise(False)
    )
)

display(silver_df)

In [0]:
# ============================================================
# 11. Duplicate groups by class
# ============================================================

duplicates_by_class = (
    silver_df
    .filter(F.col("is_duplicate_group") == True)
    .groupBy("class")
    .count()
    .orderBy(F.desc("count"))
)

display(duplicates_by_class)

In [0]:
# ============================================================
# 12. Class distribution
# ============================================================

class_distribution = (
    silver_df
    .groupBy("class")
    .count()
    .withColumn(
        "percent",
        F.round(F.col("count") / F.sum("count").over(Window.partitionBy()) * 100, 2)
    )
    .orderBy(F.desc("count"))
)

display(class_distribution)

In [0]:
# ============================================================
# 13. Save Silver table
# ============================================================

silver_table_name = "workspace.medallion_data.silver_telescope"

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table_name)
)

print(f"Silver table saved as: {silver_table_name}")

# ============================================================
# 14. Check Silver table shape
# ============================================================

silver_check = spark.table(silver_table_name)

silver_rows = silver_check.count()
silver_cols = len(silver_check.columns)

print("Silver table shape:")
print(f"Rows: {silver_rows}")
print(f"Columns: {silver_cols}")
print(f"Shape: ({silver_rows}, {silver_cols})")

display(silver_check.limit(10))
silver_check.printSchema()